# Load Data Olist ke PostgreSQL (Raw Tables)

Notebook ini me-load 9 CSV dari `data/raw/` ke 9 tabel raw di database `olist_db`.

**Prasyarat:**
- Schema sudah dibuat lewat `sql/01_schema.sql` (9 tabel kosong sudah ada)
- 9 file CSV sudah ada di `data/raw/`
- File `.env` sudah dibuat di **root repo** (bukan di `notebooks/`) berisi:
```
DB_USER=postgres
DB_PASSWORD=isi_password_postgres_kamu
DB_HOST=localhost
DB_PORT=5432
DB_NAME=olist_db
```
File ini otomatis di-ignore lewat `.gitignore`, jadi aman tidak ikut ter-push.

In [ ]:
# Install dependency kalau belum ada
# !pip install sqlalchemy psycopg2-binary python-dotenv pandas

In [5]:
import os
import getpass
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine, text, URL
from dotenv import load_dotenv

# .env ada di root repo, notebook ini di notebooks/ -> satu level di atas
load_dotenv(dotenv_path=Path("../.env"))

DB_USER = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "suksesTerus007@")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "olist_db")

if not DB_PASSWORD:
    DB_PASSWORD = getpass.getpass("Password PostgreSQL: ")

url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)

engine = create_engine(url)

with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database();"))
    print("Terhubung ke database:", result.scalar())

Terhubung ke database: olist_db


## Mapping File CSV → Nama Tabel

Sekalian daftar kolom tanggal yang perlu di-parse eksplisit sebagai `datetime`, supaya cocok dengan tipe `TIMESTAMP` di schema.

In [6]:
raw_dir = Path("../data/raw")

file_table_map = {
    "olist_customers_dataset.csv": ("customers", []),
    "olist_sellers_dataset.csv": ("sellers", []),
    "olist_products_dataset.csv": ("products", []),
    "product_category_name_translation.csv": ("product_category_translation", []),
    "olist_geolocation_dataset.csv": ("geolocation", []),
    "olist_orders_dataset.csv": ("orders", [
        "order_purchase_timestamp", "order_approved_at",
        "order_delivered_carrier_date", "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]),
    "olist_order_items_dataset.csv": ("order_items", ["shipping_limit_date"]),
    "olist_order_payments_dataset.csv": ("order_payments", []),
    "olist_order_reviews_dataset.csv": ("order_reviews", [
        "review_creation_date", "review_answer_timestamp"
    ]),
}

# Kolom numerik di 'products' kebaca float di pandas (gara-gara ada NaN),
# padahal kolom di Postgres bertipe INTEGER -> perlu dikonversi eksplisit
products_int_cols = [
    "product_name_lenght", "product_description_lenght", "product_photos_qty",
    "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"
]

## Load Semua File

Tiap tabel di-load lalu langsung diverifikasi row count-nya (CSV vs Postgres) supaya kalau ada yang gagal/kepotong, langsung ketahuan saat itu juga.

In [7]:
for filename, (table_name, date_cols) in file_table_map.items():
    filepath = raw_dir / filename
    print(f"Loading {filename} -> {table_name} ...")

    df = pd.read_csv(filepath, parse_dates=date_cols if date_cols else None)

    if table_name == "products":
        for col in products_int_cols:
            df[col] = df[col].round().astype("Int64")

    df.to_sql(
        table_name,
        engine,
        if_exists="append",
        index=False,
        method="multi",
        chunksize=10000
    )

    with engine.connect() as conn:
        db_count = conn.execute(text(f"SELECT COUNT(*) FROM {table_name}")).scalar()

    status = "OK" if db_count == len(df) else "MISMATCH!"
    print(f"  CSV: {len(df)} baris | Postgres: {db_count} baris  [{status}]")

print("\nSemua tabel selesai di-load.")

Loading olist_customers_dataset.csv -> customers ...
  CSV: 99441 baris | Postgres: 99441 baris  [OK]
Loading olist_sellers_dataset.csv -> sellers ...
  CSV: 3095 baris | Postgres: 3095 baris  [OK]
Loading olist_products_dataset.csv -> products ...
  CSV: 32951 baris | Postgres: 32951 baris  [OK]
Loading product_category_name_translation.csv -> product_category_translation ...
  CSV: 71 baris | Postgres: 71 baris  [OK]
Loading olist_geolocation_dataset.csv -> geolocation ...
  CSV: 1000163 baris | Postgres: 1000163 baris  [OK]
Loading olist_orders_dataset.csv -> orders ...
  CSV: 99441 baris | Postgres: 99441 baris  [OK]
Loading olist_order_items_dataset.csv -> order_items ...
  CSV: 112650 baris | Postgres: 112650 baris  [OK]
Loading olist_order_payments_dataset.csv -> order_payments ...
  CSV: 103886 baris | Postgres: 103886 baris  [OK]
Loading olist_order_reviews_dataset.csv -> order_reviews ...
  CSV: 99224 baris | Postgres: 99224 baris  [OK]

Semua tabel selesai di-load.


## Selesai

Kalau semua status di atas `OK` (tidak ada `MISMATCH`), data mentah sudah lengkap masuk ke `olist_db`. Langkah berikutnya di Fase 2: audit kualitas data (duplikasi, null, referential integrity antar tabel) via query SQL langsung di Postgres.